In [ ]:
import sys, numpy, pandas, sklearn, matplotlib
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.12.6 (v3.12.6:a4a2d2b0d85, Sep  6 2024, 16:08:03) [Clang 13.0.0 (clang-1300.0.29.30)]
NumPy: 2.0.2
Pandas: 2.3.3
scikit-learn: 1.6.1
Matplotlib: 3.9.4


id: S24073265

In [ ]:
import pandas as pd

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3265)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1154, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    133
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  \
listing_id          1154   1080   ALM-11147    3          NaN         NaN   
listed_date         1154    247  2026-04-18   12          NaN         NaN   
district            1154     48       Medeu  144          NaN         NaN   
rooms               1154      6           2  443          NaN         NaN   
area_

In [ ]:
df[["listed_date", "area_m2", "price_kzt", "rooms"]].sample(10, random_state=10)

,listed_date,area_m2,price_kzt,rooms
862,2026-03-03,96.7,124040000,4
849,2026-05-27,84.0,75680000,3
1105,2026-04-19,50.2,26100000,1
273,2026-06-24,41.2,"19,530,000",1
1150,2026-05-20,114.9 m²,"105,060,000",4
987,2026-04-17,85.1,38420000,3
349,2026-08-13,31.1,18670000,studio
623,2026-04-16,46.0,33390000,2
889,2026-06-06,51.5,25020000,2
157,2026-07-07,77.0,43580000,3


Q1: Pandas read some columns as text because the values are written in different formats. For example, area can include `m²`, price can have commas or a currency sign, and rooms can contain `studio`. `listed_date` is also read as text, but it should be a date. `district` and `building_type` are categories, and `listing_id` is an ID, so those can stay as text.

In [ ]:
exact_dupes = df.duplicated().sum()
print("exact copies of another row", exact_dupes)

unique_rows = df.drop_duplicates()
repeated_rows = unique_rows["listing_id"].duplicated().sum()
print("rows with a repeated listing_id after exact copies are removed:", repeated_rows)

repost_candidates = unique_rows[
    unique_rows.duplicated(subset="listing_id", keep=False)
].sort_values(["listing_id", "listed_date"])
print("listing_id values with more than one non-identical row:", repost_candidates["listing_id"].nunique())
repost_candidates

exact copies of another row 31
rows with a repeated listing_id after exact copies are removed: 43
listing_id values with more than one non-identical row: 43


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
966,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
7,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
939,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
175,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
475,ALM-10016,2026-01-20,Auezov,1,39.7,13,25,2022.0,monolith,2.80,35130000
...,...,...,...,...,...,...,...,...,...,...,...
884,ALM-11024,2026-05-04,Nauryzbay,1,35.8,11,20,2024.0,monolith,2.87,18770000
1094,ALM-11124,2026-03-18,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,19580000
191,ALM-11124,2026-04-01,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,18680000
167,ALM-11147,2026-05-26,Almaly,2,56.7,4,9,2018.0,monolith,2.86,59610000


Q2: There are 31 exact copies. After removing them, 43 rows still have a `listing_id` that appears more than once. In the pairs I checked, the apartment details are the same but the date and price are different. I will keep the row with the latest `listed_date`, because it is the newest version of the listing.

The cleaned table should have one row for each `listing_id`.

Q3:

| Column | Expected type | Valid range |
|---|---|---|
| listing_id | text ID | non-empty; unique after reposts are removed |
| listed_date | datetime | valid dates in YYYY-MM-DD format |
| district | category | one of 8: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu |
| rooms | int | 1–6 (`studio` will be 1) |
| area_m2 | float | about 15–250 m² |
| floor | int | 1 to `total_floors` |
| total_floors | int | 1–30 |
| year_built | int | 1930–2026; 0 is not a real year |
| building_type | category | panel / brick / monolith |
| ceiling_height_m | float | about 2.4–3.5 m; 0 is not valid |
| price_kzt | int | positive, roughly 10M–200M KZT |

One row in the cleaned table represents one apartment listing (`listing_id`), using its latest posting.

In [ ]:
cleaning_log = []

def log_step(operation, before, after):
    cleaning_log.append({"operation": operation, "rows_before": before, "rows_after": after})

In [ ]:
before = len(df)
df = df.drop_duplicates()
log_step("Deleting exact duplicates", before, len(df))
print(f"Was {before}, became {len(df)}")

In [ ]:
df["listed_date"] = pd.to_datetime(df["listed_date"])
df = df.sort_values("listed_date")

before = len(df)
df = df.drop_duplicates(subset="listing_id", keep="last")
log_step("Deleting reposts (kept the latest listing)", before, len(df))
print(f"Was {before}, became {len(df)}")